Group by day

In [1]:
import pandas as pd

df = pd.read_csv("data1B/final_data_1B.csv")
df["time"] = pd.to_datetime(df["time"])
df["date"] = df["time"].dt.date

sum_vars = [
    "call", "sms", "screen",
    "appCat.builtin", "appCat.communication", "appCat.entertainment",
    "appCat.finance", "appCat.game", "appCat.office",
    "appCat.other", "appCat.social", "appCat.travel",
    "appCat.unknown", "appCat.utilities", "appCat.weather"
]

mean_vars = [
    "mood", "activity",
    "circumplex.arousal", "circumplex.valence"
]

daily_parts = []

for var in df["variable"].unique():
    temp = df[df["variable"] == var].copy()
    
    if var in sum_vars:
        temp = temp.groupby(["id", "date"], as_index=False)["value"].sum()
    else:
        temp = temp.groupby(["id", "date"], as_index=False)["value"].mean()
    
    temp["variable"] = var
    daily_parts.append(temp)

df_daily_long = pd.concat(daily_parts, ignore_index=True)

df_daily = df_daily_long.pivot(
    index=["id", "date"],
    columns="variable",
    values="value"
).reset_index()

# Fill missing app/event usage with 0
app_vars = [
    "screen", "call", "sms",
    "appCat.builtin", "appCat.communication", "appCat.entertainment",
    "appCat.finance", "appCat.game", "appCat.office",
    "appCat.other", "appCat.social", "appCat.travel",
    "appCat.unknown", "appCat.utilities", "appCat.weather"
]

for col in app_vars:
    if col in df_daily.columns:
        df_daily[col] = df_daily[col].fillna(0)

df_daily = df_daily.sort_values(["id", "date"]).reset_index(drop=True)

# Reorder columns
cols = list(df_daily.columns)
for col in ["screen", "mood"]:
    if col in cols:
        cols.remove(col)

if "screen" in df_daily.columns:
    cols.append("screen")
if "mood" in df_daily.columns:
    cols.append("mood")

df_daily = df_daily[cols]

df_daily.to_csv("data1C/data_per_day.csv", index=False)

print(df.shape)
print(df_daily.shape)
print(df_daily.head())
print("Saved as data_per_day.csv")

(376478, 6)
(1973, 21)
variable       id        date  activity  appCat.builtin  appCat.communication  \
0         AS14.01  2014-02-17       NaN             0.0                   0.0   
1         AS14.01  2014-02-18       NaN             0.0                   0.0   
2         AS14.01  2014-02-19       NaN             0.0                   0.0   
3         AS14.01  2014-02-20       NaN             0.0                   0.0   
4         AS14.01  2014-02-21       NaN             0.0                   0.0   

variable  appCat.entertainment  appCat.finance  appCat.game  appCat.office  \
0                          0.0             0.0          0.0            0.0   
1                          0.0             0.0          0.0            0.0   
2                          0.0             0.0          0.0            0.0   
3                          0.0             0.0          0.0            0.0   
4                          0.0             0.0          0.0            0.0   

variable  appCat.othe

This code first sorts the data by person and date, so the next row within each person really is the next day in sequence. It then creates a new column called next_mood, which is the mood value of the following day for the same person. After that, it keeps only rows for which next_mood exists, because those are the only rows usable for supervised prediction. Rows with a current-day mood are not removed unless their next_mood is missing, so mood-containing rows are kept whenever they can still form a valid training instance. Finally, it removes rows with no usable feature information at all and saves the result as data_model_per_day.csv.

In [2]:
# Load daily dataset
df_daily = pd.read_csv("data1C/data_per_day.csv")

# Make sure date is treated as a date and data is ordered correctly
df_daily["date"] = pd.to_datetime(df_daily["date"])
df_daily = df_daily.sort_values(["id", "date"]).reset_index(drop=True)

# Create next day's mood per person
df_daily["next_mood"] = df_daily.groupby("id")["mood"].shift(-1)

# Keep rows where the target exists
# This automatically keeps all rows with a valid next-day mood,
# including rows where current-day mood is present
df_model = df_daily.dropna(subset=["next_mood"]).copy()

# Optional: if you also want to force at least one non-ID/date feature to be present
feature_cols = [col for col in df_model.columns if col not in ["id", "date", "next_mood"]]
df_model = df_model.dropna(subset=feature_cols, how="all").reset_index(drop=True)

# Save final modeling dataset
df_model.to_csv("data1C/data_usefull_per_day.csv", index=False)

print(df_daily.shape)
print(df_model.shape)
print(df_model.head())
print("Saved as data_usefull_per_day.csv")
print("Number of rows in df_daily:", len(df_daily))
print("Number of rows in df_model:", len(df_model))
print("Rows with current-day mood kept:", df_model["mood"].notna().sum())

(1973, 22)
(1268, 22)
        id       date  activity  appCat.builtin  appCat.communication  \
0  AS14.01 2014-02-25       NaN           0.000                 0.000   
1  AS14.01 2014-02-26       NaN           0.000                 0.000   
2  AS14.01 2014-03-20  0.081548         248.979              2168.229   
3  AS14.01 2014-03-21  0.134050        3139.218              6280.890   
4  AS14.01 2014-03-22  0.236880         731.429              4962.918   

   appCat.entertainment  appCat.finance  appCat.game  appCat.office  \
0                 0.000           0.000          0.0          0.000   
1                 0.000           0.000          0.0          0.000   
2               350.856           0.000          0.0          0.000   
3              1007.456          49.544          0.0        172.206   
4                93.324          21.076          0.0          0.000   

   appCat.other  ...  appCat.unknown  appCat.utilities  appCat.weather  call  \
0         0.000  ...           0

In [3]:
# Count number of days per person
days_per_person = df_daily.groupby("id")["date"].nunique()

# Print full overview
print(days_per_person)

# Optional: nicer formatted output
print("\nNumber of days per person:")
for person, days in days_per_person.items():
    print(f"{person}: {days} days")

# Optional: summary statistics
print("\nSummary statistics:")
print(days_per_person.describe())

id
AS14.01     72
AS14.02     68
AS14.03     77
AS14.05     70
AS14.06     74
AS14.07     50
AS14.08     67
AS14.09     71
AS14.12     67
AS14.13     72
AS14.14     72
AS14.15     79
AS14.16     74
AS14.17     75
AS14.19     73
AS14.20     66
AS14.23     63
AS14.24     62
AS14.25     75
AS14.26    100
AS14.27     80
AS14.28     58
AS14.29     73
AS14.30     70
AS14.31     78
AS14.32     86
AS14.33    101
Name: date, dtype: int64

Number of days per person:
AS14.01: 72 days
AS14.02: 68 days
AS14.03: 77 days
AS14.05: 70 days
AS14.06: 74 days
AS14.07: 50 days
AS14.08: 67 days
AS14.09: 71 days
AS14.12: 67 days
AS14.13: 72 days
AS14.14: 72 days
AS14.15: 79 days
AS14.16: 74 days
AS14.17: 75 days
AS14.19: 73 days
AS14.20: 66 days
AS14.23: 63 days
AS14.24: 62 days
AS14.25: 75 days
AS14.26: 100 days
AS14.27: 80 days
AS14.28: 58 days
AS14.29: 73 days
AS14.30: 70 days
AS14.31: 78 days
AS14.32: 86 days
AS14.33: 101 days

Summary statistics:
count     27.000000
mean      73.074074
std       10.6768

delete rows which dont contain mood

In [4]:

# Load your per-day dataset
df = pd.read_csv("data1C/data_per_day.csv")

# Remove rows where mood is missing
df_no_missing_mood = df.dropna(subset=["mood"]).copy()

# Save to new file
df_no_missing_mood.to_csv("data1C/no_missing_moods.csv", index=False)

# Quick check
print(df_no_missing_mood.head())
print("Saved as no_missing_moods.csv")
print("Remaining rows:", len(df_no_missing_mood))

         id        date  activity  appCat.builtin  appCat.communication  \
7   AS14.01  2014-02-26       NaN           0.000                 0.000   
8   AS14.01  2014-02-27       NaN           0.000                 0.000   
26  AS14.01  2014-03-21  0.134050        3139.218              6280.890   
27  AS14.01  2014-03-22  0.236880         731.429              4962.918   
28  AS14.01  2014-03-23  0.142741        1286.246              5237.319   

    appCat.entertainment  appCat.finance  appCat.game  appCat.office  \
7                  0.000           0.000          0.0          0.000   
8                  0.000           0.000          0.0          0.000   
26              1007.456          49.544          0.0        172.206   
27                93.324          21.076          0.0          0.000   
28                94.346          43.403          0.0          0.000   

    appCat.other  ...  appCat.travel  appCat.unknown  appCat.utilities  \
7          0.000  ...          0.000      

In [5]:
# Load dataset
df = pd.read_csv("data1C/data_per_day.csv")

# Define non-feature columns
non_feature_cols = ["id", "date", "mood"]

# All feature columns
feature_cols = [col for col in df.columns if col not in non_feature_cols]

# Count non-zero features instead of non-missing
df["num_features_present"] = (df[feature_cols] > 0).sum(axis=1)

filtered = df[
    (df["mood"].isna()) &
    (df["num_features_present"] >= 5)
]

overview = filtered.groupby("id").size()

# Print results
print("Rows per person with ≥4 features but no mood:\n")
print(overview)

# Optional: nicer format
print("\nFormatted:")
for person, count in overview.items():
    print(f"{person}: {count} rows")

# Optional: summary stats
print("\nSummary:")
print(overview.describe())

Rows per person with ≥4 features but no mood:

id
AS14.01    2
AS14.02    2
AS14.03    1
AS14.06    1
AS14.07    1
AS14.14    1
AS14.15    1
AS14.16    1
AS14.17    2
AS14.23    5
AS14.24    1
AS14.25    1
AS14.26    1
AS14.27    2
AS14.28    1
AS14.29    2
AS14.31    2
AS14.32    6
AS14.33    3
dtype: int64

Formatted:
AS14.01: 2 rows
AS14.02: 2 rows
AS14.03: 1 rows
AS14.06: 1 rows
AS14.07: 1 rows
AS14.14: 1 rows
AS14.15: 1 rows
AS14.16: 1 rows
AS14.17: 2 rows
AS14.23: 5 rows
AS14.24: 1 rows
AS14.25: 1 rows
AS14.26: 1 rows
AS14.27: 2 rows
AS14.28: 1 rows
AS14.29: 2 rows
AS14.31: 2 rows
AS14.32: 6 rows
AS14.33: 3 rows

Summary:
count    19.000000
mean      1.894737
std       1.410072
min       1.000000
25%       1.000000
50%       1.000000
75%       2.000000
max       6.000000
dtype: float64
